Original MNIST experiment
Historical experiment retained for provenance. Use the repository README for the maintained entry point.


In [ ]:
# -*- coding: utf-8 -*-
"""mnistcnn.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1TwZD5aQVBwb1y3jU5tJvd4T78FyQhs7o
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

def get_hyperparameters():

  return{
      'batch_size': 64,
      'test_batch_size':1000,
      'epochs': 5,
      'learning_rate': 0.01,
      'momentum': 0.5,
      'seed': 42,
      'log_interval': 100,
      'device': torch.device("cuda" if torch.cuda.is_available() else "cpu")
  }

config = get_hyperparameters()
torch.manual_seed(config['seed'])
device = config['device']
print(f"using device:{device}")

transform_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    './data',
    train=True,
    download=True,
    transform=transform_pipeline
)
test_dataset = datasets.MNIST(
    './data',
    train=False,
    download=True,
    transform=transform_pipeline
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=config['test_batch_size'],
    shuffle=False
)

def train(epoch , model , optimizer, log_interval, train_loader, device):
  model.train()
  for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device)
    optimizer.zero_grad()
    output = model(data)
    loss = F.nll_loss(output, target)
    loss.backward()
    optimizer.step()

    if batch_idx % log_interval == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                  f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

def test(model, test_loader, device):
    model.eval()
    test_loss = 0
    correct = 0

def test(model, test_loader, device):
    model.eval()
    test_loss = 0
    correct = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)

            test_loss += F.nll_loss(output, target, reduction='sum').item()

            pred = output.argmax(dim=1, keepdim=True)

            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = 100. * correct / len(test_loader.dataset)

    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} '
          f'({test_accuracy:.2f}%)\n')

    return test_loss, test_accuracy

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(0.25)

        dummy_input = torch.randn(1, 1, 28, 28)
        dummy_output = self.pool(F.relu(self.conv1(dummy_input)))
        dummy_output = self.pool(F.relu(self.conv2(dummy_output)))
        flattened_size = torch.flatten(dummy_output, 1).shape[1]

        self.fc1 = nn.Linear(flattened_size, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

if __name__ == '__main__':
    cnn_model = Net().to(device)

    optimizer = optim.SGD(
        cnn_model.parameters(),
        lr=config['learning_rate'],
        momentum=config['momentum']
    )

    test_losses = []
    test_accuracies = []

    for epoch in range(1, config['epochs'] + 1):
        train(epoch, cnn_model, optimizer, config['log_interval'], train_loader, device)
        loss, accuracy = test(cnn_model, test_loader, device)
        test_losses.append(loss)
        test_accuracies.append(accuracy)

    print("Training Complete!")

    plt.figure(figsize=(10, 5))
    plt.plot(range(1, config['epochs'] + 1), test_accuracies, marker='o', linestyle='-')
    plt.title('Test Accuracy per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.grid(True)
    plt.show()